# RevisitingCIL for IP102 — Kaggle Notebook

**Input bắt buộc** (Add ngoài cửa sổ Input phải):

1. **Code RevisitingCIL** (repo nta2112/RevisitingCIL-for-IP102, zip bỏ `.git`) làm private dataset
2. **IP102 dataset** (folder chứa `train.json`/`test.json`/`val.json` + `VOC2007/VOC2007/JPEGImages`) — dataset bạn vẫn dùng cho iCaRL

**Tuỳ biến qua env var (bật Settings → Notebook options → Environment variables):**

- `IP102_MODEL`: `simplecil` (mặc định) hoặc `aper_finetune`
- `IP102_MAX_TASKS`: chạy N task đầu để test nhanh (VD `1`)
- `IP102_MEMORY_SIZE`: tổng exemplar (mặc định 2000)

**Kết quả:** `/kaggle/working/logs/simplecil/ip102/...` — `results.csv`, `history.json`, `*.log`

In [ ]:
import os, sys, torch, timm
print('python', sys.version.split()[0])
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| gpus', torch.cuda.device_count())
print('timm', timm.__version__)

In [ ]:
import os

def find_dir_with_file(base, fname, maxdepth=10):
    base = os.path.abspath(base)
    for dp, dn, fn in os.walk(base):
        depth = dp[len(base):].count(os.sep)
        if depth > maxdepth:
            dn[:] = []
            continue
        if fname in fn:
            return dp
    return None

DATA_ROOT = os.environ.get('IP102_DATA_ROOT') or find_dir_with_file('/kaggle/input', 'train.json')
CODE_SRC  = os.environ.get('IP102_CODE_DIR')  or find_dir_with_file('/kaggle/input', 'main.py')

print('DATA_ROOT:', DATA_ROOT)
print('CODE_SRC :', CODE_SRC)
assert DATA_ROOT and os.path.exists(os.path.join(DATA_ROOT, 'train.json')), 'Khong tim thay IP102 dataset'
assert CODE_SRC  and os.path.exists(os.path.join(CODE_SRC, 'main.py')),  'Khong tim thay code RevisitingCIL'

In [ ]:
import os, shutil

WORK_CODE = '/kaggle/working/RevisitingCIL'
if not os.path.exists(WORK_CODE):
    print('Copying code to', WORK_CODE)
    shutil.copytree(CODE_SRC, WORK_CODE,
                    ignore=shutil.ignore_patterns('*.pyc', '__pycache__', '.git'))
print('WORK_CODE:', WORK_CODE)

In [ ]:
import json, os, subprocess, sys

MODEL = os.environ.get('IP102_MODEL', 'simplecil').strip().lower()
MAX_TASKS = int(os.environ.get('IP102_MAX_TASKS', 0))
MEMORY_SIZE = int(os.environ.get('IP102_MEMORY_SIZE', 2000))

CONFIG_REL = {
    'simplecil':     'exps/simplecil/ip102_7_6_6_6_vit-b_simplecil.json',
    'aper_finetune': 'exps/aper_finetune/ip102_7_6_6_6_vit-b_finetune.json',
}[MODEL]

cfg_path = os.path.join(WORK_CODE, CONFIG_REL)
cfg = json.load(open(cfg_path, encoding='utf-8'))
cfg['memory_size'] = MEMORY_SIZE

if MAX_TASKS > 0:
    n = min(MAX_TASKS, 25 // 7 + 1)
    if n <= 1:
        cfg['increment'] = 25 - 7
    else:
        cfg['increment'] = -(-(25 - 7) // (n - 1))
    cfg_path = '/kaggle/working/config_tmp.json'
    json.dump(cfg, open(cfg_path, 'w', encoding='utf-8'), indent=2)

print('model:', MODEL, '| memory_size:', MEMORY_SIZE, '| max_tasks:', MAX_TASKS or 'all')
print('config:', cfg_path)
print('running main.py ...')
sys.stdout.flush()

ret = subprocess.call([sys.executable, 'main.py', '--config', cfg_path], cwd=WORK_CODE)
print('main.py return code:', ret)

In [ ]:
import glob, os
import pandas as pd

csvs = glob.glob('/kaggle/working/logs/**/results.csv', recursive=True)
if not csvs:
    print('Chua co results.csv (train chua xong hoac that bai)')
else:
    for p in sorted(csvs):
        print('===', p)
        try:
            print(pd.read_csv(p).to_string(index=False))
        except Exception as e:
            print('khong doc duoc:', e)